In [2]:
pip install selenium


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import time
import json
import pandas as pd
import logging

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


with open("config.json", "r", encoding="utf-8") as f:
    config = json.load(f)


logger = logging.getLogger(__name__)

if config["logging_enabled"]:
    handler = logging.FileHandler("app_2.log", encoding="utf-8")
    logger.addHandler(handler)

    formatter = logging.Formatter("%(levelname)s | %(message)s")
    handler.setFormatter(formatter)

    if config["logging_level"] == "DEBUG":
        logger.setLevel(logging.DEBUG)
    elif config["logging_level"] == "ERROR":
        logger.setLevel(logging.ERROR)
    else:
        logger.setLevel(logging.INFO)



def parse_all_trains(city, driver, trip_date):
    rows = []
    train_containers = driver.find_elements(By.CSS_SELECTOR, ".wg-train-container")
    logger.info(f"Найдено поездов для {city} на {trip_date}: {len(train_containers)}")
    for container in train_containers:
        train = {}
        num_elem = container.find_elements(By.CSS_SELECTOR, ".wg-train-info__number-link")
        train['number'] = num_elem[0].text if num_elem else ""
        train['name'] = num_elem[1].text if len(num_elem) > 1 else ""
        train['city'] = city
        direction = container.find_element(By.CSS_SELECTOR, ".wg-train-info__direction")
        train['route'] = direction.text
        
        times = container.find_elements(By.CSS_SELECTOR, ".wg-track-info__time")
        train['departure_time'] = times[0].text if times else ""
        train['arrival_time'] = times[1].text if len(times) > 1 else ""
        
        stations = container.find_elements(By.CSS_SELECTOR, ".wg-track-info__station")
        train['from_station'] = stations[0].text if stations else ""
        train['to_station'] = stations[1].text if len(stations) > 1 else ""
        
        rating = container.find_elements(By.CSS_SELECTOR, ".wg-train-rating__value")
        train['rating'] = rating[0].text if rating else ""
        
        wagon_items = container.find_elements(By.CSS_SELECTOR, ".wg-wagon-type__item")

        time_in_road = container.find_elements(By.CSS_SELECTOR, ".wg-track-info__duration > span")
        train['time_in_road'] = time_in_road[0].text if time_in_road else ""
        if wagon_items:
            for wagon in wagon_items:
                row = train.copy()
                row['date'] = trip_date
                row['wagon_type'] = wagon.find_element(By.CSS_SELECTOR, ".wg-wagon-type__title").text
                row['wagon_seats'] = wagon.find_element(By.CSS_SELECTOR, ".wg-wagon-type__available-seats").text
                row['wagon_price'] = wagon.find_element(By.CSS_SELECTOR, ".wg-wagon-type__price-value").text
                rows.append(row)
        else:
            row = train.copy()
            row['date'] = trip_date
            row['wagon_type'] = ""
            row['wagon_seats'] = ""
            row['wagon_price'] = ""
            rows.append(row)
    
    return rows


all_rows = []
citys = [
    "Санкт-Петербург",
    "Новосибирск",
    "Екатеринбург",
    "Казань",
    "Нижний Новгород",
    "Челябинск",
    "Красноярск",
    "Самара",
    "Уфа",
    "Ростов-на-Дону",
    "Омск",
    "Краснодар",
    "Воронеж",
    "Пермь",
    "Волгоград",
    "Саратов",
    "Тюмень",
    "Ижевск",
    "Барнаул",
    "Ульяновск",
    "Иркутск",
    "Хабаровск",
    "Ярославль",
    "Владивосток",
    "Махачкала",
    "Томск",
    "Оренбург",
    "Кемерово",
    "Новокузнецк",
    "Рязань",
    "Астрахань",
    "Набережные Челны",
    "Пенза",
    "Липецк",
    "Киров",
    "Чебоксары",
    "Калининград",
    "Тула",
    "Курск",
    "Ставрополь",
    "Сочи",
    "Улан-Удэ",
    "Тверь",
    "Магнитогорск",
    "Иваново",
    "Брянск",
    "Белгород",
    "Сургут",
    "Владимир",
    "Нижний Тагил",
    "Архангельск",
    "Чита",
    "Калуга",
    "Смоленск",
    "Волжский",
    "Якутск",
    "Саранск",
    "Череповец",
    "Курган",
    "Вологда",
    "Орёл",
    "Грозный",
    "Мурманск",
    "Тамбов",
    "Петрозаводск",
    "Стерлитамак",
    "Кострома",
    "Нижневартовск",
    "Йошкар-Ола",
    "Новороссийск",
    "Комсомольск-на-Амуре",
    "Таганрог",
    "Сыктывкар",
    "Нальчик",
    "Шахты",
    "Дзержинск",
    "Орск",
    "Благовещенск",
    "Братск",
    "Энгельс",
    "Мытищи",
    "Люберцы",
    "Королёв",
    "Прокопьевск",
    "Старый Оскол",
    "Ангарск",
    "Великий Новгород",
    "Псков",
    "Бийск",
    "Рыбинск",
    "Армавир",
    "Северодвинск",
    "Балаково",
    "Абакан",
    "Норильск",
    "Вятские-Поляны",
    "Кировск",
    "Симферополь",
    "Севастополь",
    "Евпатория"
]

for city in citys[:25]:
    for day in range(1, 31):
        driver = None
        try:
            logger.info(f"Начало парсинга: {city}, день {day}")

            URL = "https://xn----btbhgbpv1d7d.xn--p1ai/kupit-rzhd-bilety/"
            FROM_CITY = "Москва"

            options = webdriver.ChromeOptions()
            driver = webdriver.Chrome(options=options)
            logger.info("Запуск браузера")
            driver.get(URL)
            logger.info(f"Открыт сайт для {city}, день {day}")

            WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.TAG_NAME, "body"))
            )

            element = driver.find_element(
                By.CSS_SELECTOR,
                "#ufs-railway-app > div > div > div > div.uwg-search__way > div.uwg-search__from > div > div > input"
            )
            element.click()
            time.sleep(0.3)
            element.send_keys(Keys.CONTROL, "a")
            element.send_keys(Keys.BACKSPACE)
            time.sleep(0.2)
            element.send_keys(FROM_CITY)
            element.send_keys(Keys.DOWN)
            element.send_keys(Keys.ENTER)

            element = driver.find_element(
                By.CSS_SELECTOR,
                "#ufs-railway-app > div > div > div > div.uwg-search__way > div.uwg-search__to > div > div > input"
            )
            element.click()
            time.sleep(0.3)
            element.send_keys(Keys.CONTROL, "a")
            element.send_keys(Keys.BACKSPACE)
            time.sleep(0.2)
            element.send_keys(city)
            time.sleep(1)
            element.send_keys(Keys.DOWN)
            time.sleep(0.3)
            element.send_keys(Keys.ENTER)

            try:
                day_element = WebDriverWait(driver, 5).until(
                    EC.element_to_be_clickable((
                        By.XPATH,
                        f"//span[contains(@class, 'uwg-datepicker__month') and contains(text(), 'июнь')]/ancestor::div[contains(@class, 'uwg-datepicker__row')]//a[contains(@class, 'uwg-datepicker__day') and text()='{day}']"
                    ))
                )
                day_element.click()
                logger.info(f"Дата выбрана с первого раза: {day}.06.2026")

            except Exception as e:
                logger.warning(f"Не удалось выбрать дату сразу: {city}, день {day}, ошибка: {e}")

                driver.find_element(
                    By.CSS_SELECTOR,
                    "#ufs-railway-app > div > div > div > div.uwg-search__date > div:nth-child(1) > div"
                ).click()
                time.sleep(1)

                day_element = WebDriverWait(driver, 5).until(
                    EC.element_to_be_clickable((
                        By.XPATH,
                        f"//span[contains(@class, 'uwg-datepicker__month') and contains(text(), 'июнь')]/ancestor::div[contains(@class, 'uwg-datepicker__row')]//a[contains(@class, 'uwg-datepicker__day') and text()='{day}']"
                    ))
                )
                day_element.click()
                logger.info(f"Дата выбрана со второй попытки: {day}.06.2026")

            search_btn = driver.find_element(
                By.CSS_SELECTOR,
                "#ufs-railway-app > div > div > div > div.uwg-search__submit > button"
            )
            search_btn.click()

            time.sleep(5)
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, ".wg-train-container"))
            )
            rows = parse_all_trains(city, driver, f"{day:02d}.06.2026")
            all_rows.extend(rows)

            logger.info(f"Успешно спарсено строк: {len(rows)} для {city}, {day}.06.2026")

        except Exception as e:
            print(city, day)
            logger.error(f"Ошибка при парсинге {city}, день {day}: {e}")
        finally:
            if driver is not None:
                driver.quit()


df = pd.DataFrame(all_rows)
df.to_csv("result_3.csv", index=False, encoding="utf-8-sig")
logger.info(f"Файл result_3.csv сохранен. Всего строк: {len(df)}")
print("Готово")

Челябинск 14
Пермь 4
Пермь 8
Пермь 10
Пермь 11
Пермь 15
Пермь 16
Пермь 17
Пермь 18
Пермь 19
Пермь 23
Пермь 24
Пермь 29
Пермь 30
Барнаул 3
Хабаровск 22
Хабаровск 29
Ярославль 14
Ярославль 19
Ярославль 21
Готово


In [25]:
df = pd.DataFrame(all_rows)
df.to_csv('result_5.csv')

In [26]:
df

,number,name,city,route,departure_time,arrival_time,from_station,to_station,rating,time_in_road,date,wagon_type,wagon_seats,wagon_price
0,№ 022А,Ночной Экспресс,Санкт-Петербург,Москва → Санкт-Петербург,00:25,08:53,Ленинградский вокзал,Московский вокзал,9.8,08 ч 28 м,01.06.2026,Купе,Мест: 285,"5 828,6 R"
1,№ 022А,Ночной Экспресс,Санкт-Петербург,Москва → Санкт-Петербург,00:25,08:53,Ленинградский вокзал,Московский вокзал,9.8,08 ч 28 м,01.06.2026,СВ,Мест: 15,"15 980,2 R"
2,№ 016А,Арктика,Санкт-Петербург,Москва → Санкт-Петербург → Мурманск,00:46,09:15,Ленинградский вокзал,Ладожский вокзал,8.1,08 ч 29 м,01.06.2026,Плацкарт,Мест: 32,"3 577,8 R"
3,№ 016А,Арктика,Санкт-Петербург,Москва → Санкт-Петербург → Мурманск,00:46,09:15,Ленинградский вокзал,Ладожский вокзал,8.1,08 ч 29 м,01.06.2026,Купе,Мест: 35,"5 761,7 R"
4,№ 016А,Арктика,Санкт-Петербург,Москва → Санкт-Петербург → Мурманск,00:46,09:15,Ленинградский вокзал,Ладожский вокзал,8.1,08 ч 29 м,01.06.2026,СВ,Мест: 12,"16 153,5 R"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,№ 030У,,Санкт-Петербург,Белгород → Москва → Санкт-Петербург,23:52,09:25,ВК Восточный,Московский вокзал,9.4,09 ч 33 м,01.06.2026,Сидячий,Мест: 97,"3 015,2 R"
162,№ 030У,,Санкт-Петербург,Белгород → Москва → Санкт-Петербург,23:52,09:25,ВК Восточный,Московский вокзал,9.4,09 ч 33 м,01.06.2026,Купе,Мест: 320,"5 138,5 R"
163,№ 030У,,Санкт-Петербург,Белгород → Москва → Санкт-Петербург,23:52,09:25,ВК Восточный,Московский вокзал,9.4,09 ч 33 м,01.06.2026,СВ,Мест: 26,"17 132,8 R"
164,№ 002А,Красная Стрела,Санкт-Петербург,Москва → Санкт-Петербург,23:55,07:55,Ленинградский вокзал,Московский вокзал,9.4,08 ч 00 м,01.06.2026,Купе,Мест: 142,"5 725,4 R"
